In [1]:
import os
import pandas as pd
from biopandas.pdb import PandasPdb
import yaml
import numpy as np

In [2]:


config_path = os.path.join(os.path.dirname(os.getcwd()), 'config.yaml')

print(config_path)

with open(config_path,'r') as yaml_file:
    config = yaml.safe_load(yaml_file)

raw_data_folder = os.path.join(config['top_level_folder'],'data/raw')

raw_train_data_folder = os.path.join(raw_data_folder,'train')
raw_test_data_folder = os.path.join(raw_data_folder,'test')

train_pdb_filenames = os.listdir(raw_train_data_folder)
print(f'num train files: {len(train_pdb_filenames)}')

test_pbd_filenames = os.listdir(raw_test_data_folder)
print(f'num test files: {len(test_pbd_filenames)}')

train_pdb_filepaths = [os.path.join(raw_train_data_folder, filename) for filename in train_pdb_filenames if filename.endswith('.pdb')]
single_test_file = train_pdb_filepaths[0]

print(train_pdb_filepaths[0])

/home/leon/Documents/github_repos/deep_origin_take_home_assignment/deep_origin/src/config.yaml
num train files: 13368
num test files: 500
/home/leon/Documents/github_repos/deep_origin_take_home_assignment/deep_origin/data/raw/train/5D17.pdb


In [4]:

def create_df_from_folder(folder_path):

    rows = []
    for fname in os.listdir(folder_path):
        if not fname.endswith(".pdb"):
            continue

        path = os.path.join(folder_path, fname)
        ppdb = PandasPdb().read_pdb(path)
        # print(ppdb)
        # Extract ATOM records
        # ca_df = ppdb.df
        df = ppdb.df['ATOM']

        # Filter for alpha carbons (CA)
        # ca_df = df[df['atom_name'] == 'CA'].copy()

        # Add filename for reference
        # ca_df['pdb_id'] = os.path.splitext(fname)[0]

        rows.append(df)

        break

    # Concatenate all into one DataFrame
    # all_ca = pd.concat(rows, ignore_index=True)

    # Optional: keep only useful columns
    # all_ca = all_ca[['pdb_id', 'chain_id', 'residue_number',
    #                  'residue_name', 'atom_name', 'x_coord', 'y_coord', 'z_coord']]
        
    return ca_df

# # Example usage
# pdb_dir = "path/to/your/folder"
ca_table = create_df_from_folder(raw_train_data_folder)
ca_table.head()

NameError: name 'ca_df' is not defined

In [7]:
ca_table.columns

Index(['record_name', 'atom_number', 'blank_1', 'atom_name', 'alt_loc',
       'residue_name', 'blank_2', 'chain_id', 'residue_number', 'insertion',
       'blank_3', 'x_coord', 'y_coord', 'z_coord', 'occupancy', 'b_factor',
       'blank_4', 'segment_id', 'element_symbol', 'charge', 'line_idx',
       'pdb_id'],
      dtype='object')

In [10]:
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Dict, Optional
import pandas as pd
from biopandas.pdb import PandasPdb

# Standard residues for quick filtering (extend if you like)
STD_RES = {
    "ALA","ARG","ASP","CYS","CYX","GLN","GLU","GLY","HIS","HIE","ILE","LEU",
    "LYS","MET","ASN","PHE","PRO","SEC","SER","THR","TRP","TYR","VAL"
}

def _fast_header_scan(pdb_path: str):
    """Cheap scan for number of models, experimental method, and resolution."""
    n_models = 0
    exp_method = None
    resolution = None

    with open(pdb_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if line.startswith("MODEL "):
                n_models += 1
            elif line.startswith("EXPDTA") and exp_method is None:
                exp_method = line[10:].strip()
            elif line.startswith("REMARK   2 RESOLUTION.") and resolution is None:
                # Usually: REMARK   2 RESOLUTION.    1.80 ANGSTROMS.
                parts = line.split()
                for tok in parts:
                    try:
                        resolution = float(tok)
                        break
                    except ValueError:
                        pass
            # tiny early exit
            if exp_method is not None and resolution is not None and n_models > 0:
                # keep scanning in case more MODEL lines exist
                continue
    if n_models == 0:
        n_models = 1  # typical single-model PDBs omit MODEL records
    return n_models, exp_method, resolution

def summarize_pdb_fast(pdb_path: str) -> Optional[Dict]:
    try:
        ppdb = PandasPdb().read_pdb(pdb_path)

        atoms = ppdb.df.get("ATOM")
        if atoms is None or atoms.empty:
            return None

        # Optional: keep only standard amino acids
        atoms_std = atoms[atoms["residue_name"].isin(STD_RES)]

        # Vectorised counts
        n_atoms = len(atoms_std)
        n_chains = atoms_std["chain_id"].nunique()
        # residue identity = chain_id + residue_number + insertion code (if present)
        if "insertion" in atoms_std.columns:
            n_residues = atoms_std.drop_duplicates(["chain_id", "residue_number", "insertion"]).shape[0]
        else:
            n_residues = atoms_std.drop_duplicates(["chain_id", "residue_number"]).shape[0]
        n_ca = (atoms_std["atom_name"] == "CA").sum()

        # Header stats (fast scan; avoids heavy biopython parsing)
        n_models, exp_method, resolution = _fast_header_scan(pdb_path)

        return {
            "file": os.path.basename(pdb_path),
            "models": n_models,
            "chains": int(n_chains),
            "residues": int(n_residues),
            "atoms": int(n_atoms),
            "ca_atoms": int(n_ca),
            "exp_method": exp_method,
            "resolution": resolution,
        }
    except Exception:
        return None

def summarize_folder(pdb_files_to_check: List[str], max_workers: int = None) -> pd.DataFrame:
    if not pdb_files_to_check:
        return pd.DataFrame(columns=[
            "file","models","chains","residues","atoms","ca_atoms","exp_method","resolution"
        ])

    # Many small files → threads help; heavy parsing → consider processes
    if max_workers is None:
        cpu = os.cpu_count() or 4
        max_workers = min(64, cpu * 5)

    results: List[Dict] = []
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = {ex.submit(summarize_pdb_fast, p): p for p in pdb_files_to_check}
        for fut in as_completed(futs):
            rec = fut.result()
            if rec:
                results.append(rec)

    return pd.DataFrame(results)

df = summarize_folder(train_pdb_filepaths[0:1000]) 


#

       file  models  chains  residues  atoms  ca_atoms         exp_method  \
0  4WXT.pdb       1       1       108    829       111  X-RAY DIFFRACTION   
1  4MAI.pdb       1       1       187   1381       188  X-RAY DIFFRACTION   
2  1O0T.pdb       1       1       455   3356       455  X-RAY DIFFRACTION   
3  3QKJ.pdb       1       4       524   4125       526  X-RAY DIFFRACTION   
4  1MUF.pdb       1       1       251   1967       251  X-RAY DIFFRACTION   

   resolution  
0         2.0  
1         2.0  
2         2.0  
3         2.0  
4         2.0  


In [13]:
df.tail()

,file,models,chains,residues,atoms,ca_atoms,exp_method,resolution
995,7P3J.pdb,1,2,694,5525,698,X-RAY DIFFRACTION,2.0
996,6DEH.pdb,1,2,650,5029,653,X-RAY DIFFRACTION,2.0
997,6JQH.pdb,1,2,1017,8122,1017,X-RAY DIFFRACTION,2.0
998,4CT5.pdb,1,2,750,11613,750,X-RAY DIFFRACTION,2.0
999,3AIE.pdb,1,8,6752,53279,6752,X-RAY DIFFRACTION,2.0


In [11]:
df.he

models
1     999
16      1
Name: count, dtype: int64

Exploring single files


In [4]:
def load_single_pdb_file(file_path):

    ppdb = PandasPdb().read_pdb(file_path)
    df = ppdb.df['ATOM']
    cols_to_drop = [col for col in df.columns if 'blank' in col]
    df = df.drop(columns=cols_to_drop)

    #only need CA entries
    ca_df = df[df["atom_name"] == "CA"]

    #need to return separate data per chain
    
    
    return ca_df

test_df = load_single_pdb_file(single_test_file)
test_df.head()

,record_name,atom_number,atom_name,alt_loc,residue_name,chain_id,residue_number,insertion,x_coord,y_coord,z_coord,occupancy,b_factor,segment_id,element_symbol,charge,line_idx
1,ATOM,2,CA,,GLN,A,376,,-7.191,22.470,81.074,1.0,101.24,,C,NaN,3145
8,ATOM,9,CA,,ASP,A,377,,-7.361,26.271,80.881,1.0,94.53,,C,NaN,3159
16,ATOM,17,CA,,ALA,A,378,,-8.479,27.434,77.426,1.0,84.91,,C,NaN,3175
21,ATOM,22,CA,,THR,A,379,,-8.779,31.176,78.123,1.0,82.07,,C,NaN,3185
28,ATOM,29,CA,,ASN,A,380,,-7.248,33.500,75.522,1.0,74.49,,C,NaN,3199


In [48]:
test_pdb = PandasPdb().read_pdb(single_test_file)
test_df = test_pdb
test_df.head()

,record_name,atom_number,blank_1,atom_name,alt_loc,residue_name,blank_2,chain_id,residue_number,insertion,...,x_coord,y_coord,z_coord,occupancy,b_factor,blank_4,segment_id,element_symbol,charge,line_idx
0,ATOM,1,,N,,GLN,,A,376,,...,-8.042,21.798,82.050,1.0,106.81,,,N,NaN,3143
1,ATOM,2,,CA,,GLN,,A,376,,...,-7.191,22.470,81.074,1.0,101.24,,,C,NaN,3145
2,ATOM,3,,C,,GLN,,A,376,,...,-7.713,23.854,80.695,1.0,99.78,,,C,NaN,3147
3,ATOM,4,,O,,GLN,,A,376,,...,-8.733,23.983,80.018,1.0,100.30,,,O,NaN,3149
4,ATOM,5,,CB,,GLN,,A,376,,...,-7.036,21.624,79.812,1.0,105.08,,,C,NaN,3151


In [12]:
test_df.chain_id.unique()

array(['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M'],
      dtype=object)

In [13]:
def convert_ca_df_into_per_chain_data(df):

    unique_chains = df.chain_id.unique()
    dfs_per_chain = []
    for chain in unique_chains:
        temp_df = df.copy()
        temp_df = temp_df[temp_df['chain_id'] == chain]
        temp_df['res_num_reindexed'] = range(1, len(temp_df) + 1)
        dfs_per_chain.append(temp_df)


    return dfs_per_chain

test = convert_ca_df_into_per_chain_data(test_df)


In [17]:
test[0].head(5)

,record_name,atom_number,atom_name,alt_loc,residue_name,chain_id,residue_number,insertion,x_coord,y_coord,z_coord,occupancy,b_factor,segment_id,element_symbol,charge,line_idx,res_num_reindexed
1,ATOM,2,CA,,GLN,A,376,,-7.191,22.470,81.074,1.0,101.24,,C,NaN,3145,1
8,ATOM,9,CA,,ASP,A,377,,-7.361,26.271,80.881,1.0,94.53,,C,NaN,3159,2
16,ATOM,17,CA,,ALA,A,378,,-8.479,27.434,77.426,1.0,84.91,,C,NaN,3175,3
21,ATOM,22,CA,,THR,A,379,,-8.779,31.176,78.123,1.0,82.07,,C,NaN,3185,4
28,ATOM,29,CA,,ASN,A,380,,-7.248,33.500,75.522,1.0,74.49,,C,NaN,3199,5


In [20]:
import numpy as np
from scipy.spatial.distance import pdist, squareform


def convert_coords_to_distance_matrix_single_chain(single_chain_df):

    
    n_residues = len(df)
    dist_matrix = np.zeros((n_residues, n_residues))

    all_coords = np.asarray(single_chain_df[['x_coord','y_coord','z_coord']])

    return squareform(pdist(all_coords, metric='euclidean')).astype(np.float32, copy=False)


dist_matrix = convert_coords_to_distance_matrix_single_chain(test[0])

In [22]:
test[0].head(5)

,record_name,atom_number,atom_name,alt_loc,residue_name,chain_id,residue_number,insertion,x_coord,y_coord,z_coord,occupancy,b_factor,segment_id,element_symbol,charge,line_idx,res_num_reindexed
1,ATOM,2,CA,,GLN,A,376,,-7.191,22.470,81.074,1.0,101.24,,C,NaN,3145,1
8,ATOM,9,CA,,ASP,A,377,,-7.361,26.271,80.881,1.0,94.53,,C,NaN,3159,2
16,ATOM,17,CA,,ALA,A,378,,-8.479,27.434,77.426,1.0,84.91,,C,NaN,3175,3
21,ATOM,22,CA,,THR,A,379,,-8.779,31.176,78.123,1.0,82.07,,C,NaN,3185,4
28,ATOM,29,CA,,ASN,A,380,,-7.248,33.500,75.522,1.0,74.49,,C,NaN,3199,5


In [34]:
def make_residue_seq_per_chain(chain_df):


    chain_df['residue_single_letter'] = chain_df['residue_name'].map(residue_to_one_letter)

    

    return chain_df['residue_single_letter'].to_list()

res_seq = make_residue_seq_per_chain(test[0])
print(res_seq)

['Q', 'D', 'A', 'T', 'N', 'Y', 'N', 'X', 'I', 'F', 'A', 'N', 'R', 'F', 'A', 'A', 'F', 'D', 'E', 'L', 'L', 'X', 'I', 'L', 'K', 'T', 'K', 'F', 'A', 'C', 'R', 'V', 'L', 'F', 'E', 'E', 'T', 'L', 'V', 'L', 'P', 'K', 'V', 'G', 'R', 'X', 'R', 'L', 'H', 'L', 'C', 'K', 'D', 'G', 'X', 'P', 'R', 'V', 'I', 'K', 'A', 'V', 'G', 'V', 'Q', 'R', 'N', 'G', 'X', 'E', 'F', 'V', 'L', 'L', 'E', 'V', 'D', 'A', 'X', 'D', 'G', 'V', 'V', 'K', 'K', 'L', 'L', 'X', 'X', 'T', 'T', 'K', 'V', 'L', 'X', 'G', 'V', 'D', 'X', 'E', 'T', 'W', 'R', 'N', 'D', 'F', 'E', 'K', 'I', 'R', 'R', 'G', 'V', 'V', 'K', 'X', 'X', 'L', 'N', 'W', 'P', 'N', 'X', 'L', 'F', 'D', 'Q', 'L', 'Y', 'G', 'Q', 'D', 'G', 'H', 'R', 'G', 'V', 'N', 'H', 'P', 'K', 'G', 'L', 'G', 'E', 'L', 'Q', 'V', 'X', 'R', 'E', 'D', 'E', 'G', 'W', 'A', 'E', 'R', 'V', 'V', 'R', 'E']


In [30]:
RES3_TO_RES1 = {
    "ALA": "A",
    "ARG": "R",
    "ASP": "D",
    "CYS": "C",
    "CYX": "C",  
    "GLN": "Q",
    "GLU": "E",
    "GLY": "G",
    "HIS": "H",
    "HIE": "H",  
    "ILE": "I",
    "LEU": "L",
    "LYS": "K",
    "MET": "M",
    "ASN": "N",
    "PHE": "F",
    "PRO": "P",
    "SEC": "U",  
    "THR": "T",
    "TRP": "W",
    "TYR": "Y",
    "VAL": "V",
}

def residue_to_one_letter(residue_name: str) -> str:
    """
    Convert a 3-letter residue name to its 1-letter amino acid code.
    Returns 'X' for unknown or non-standard residues.
    """
    residue_name = residue_name.strip().upper()
    return RES3_TO_RES1.get(residue_name, "X")

Processing npy files

In [9]:
test_file = '/home/leon/Documents/github_repos/deep_origin_take_home_assignment/deep_origin/data/processed/train_data_processed/106M.npy'
test_data = np.load(test_file, allow_pickle=True)

print(test_data)

[[0
  'MVLXEGEWQLVLHVWAKVEADVAGHGQDILIRLFKXHPETLEKFDRFKHLKTEAEMKAXEDLKKHGVTFLTALGAILKKKGHHEAELKPLAQXHATKHKIPIKYLEFIXEAIIHVLHXRHPGNFGADAQGAMNKALELFRKDIAAKYKELGYQG'
  array([[0, 1, 1, ..., 0, 0, 0],
         [1, 0, 1, ..., 0, 0, 0],
         [1, 1, 0, ..., 0, 0, 0],
         ...,
         [0, 0, 0, ..., 0, 1, 1],
         [0, 0, 0, ..., 1, 0, 1],
         [0, 0, 0, ..., 1, 1, 0]], shape=(154, 154), dtype=uint8)]]


In [4]:
import torch

print("PyTorch available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"Device {i}: {torch.cuda.get_device_name(i)}")


PyTorch available: True
GPU count: 1
Device 0: NVIDIA GeForce RTX 2050


In [5]:
with open(config_path,'r') as yaml_file:
        config = yaml.safe_load(yaml_file)

train_data_pkl_path = config['train_data_combined_filepath']

training_df = pd.read_pickle(train_data_pkl_path)



In [8]:
training_df['seq_length'] = training_df['residue_sequence'].apply(len)
training_df['seq_length'].value_counts()

seq_length
129    92
126    70
162    63
130    62
330    61
       ..
873     1
69      1
609     1
849     1
918     1
Name: count, Length: 779, dtype: int64

In [6]:
training_df.head()

,filename,chain_number,residue_sequence,contact_map
0,3GDE.npy,1,GXHMLFAEFAEFCERLEKIXXTLELTARIAAFLQKIEDERDLYDVV...,"[[0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
1,4QFO.npy,1,QGLVYCAEANPVXFNPQVTTTGXTIDIIANQLYDRLIXIDPVTAEF...,"[[0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
2,3MWG.npy,1,KRVVTLYQGATDVAVXLGVKPVGAVEXWTQKPKFEYIKNDLKDTKI...,"[[0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
3,4S3L.npy,1,DDNXAITKANGENNAVVKINKTLNIAEGITTPTATFTFKFTEKTGQ...,"[[0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
4,4GVE.npy,1,DIGXEFEGQELIVRAAVXELDPXNTIWLDIEGPPTDPVELALYQPA...,"[[0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."


In [16]:
#find rows larger than 354

large_seq_df = training_df[training_df['seq_length'] > 600]
len(large_seq_df)

527

In [8]:
test_embed = '/home/leon/Documents/github_repos/deep_origin_take_home_assignment/deep_origin/data/embeds_preds/train/1FY7_ch_1_embeds_preds.npz'
test_array = np.load(test_embed)
print(f"Keys in npz file: {test_array.files}")
print(f"Embeddings shape: {np.shape(test_array['embeddings'])}")
print(f"Contact map shape: {np.shape(test_array['contact_map'])}")

Keys in npz file: ['embeddings', 'contact_map']
Embeddings shape: (273, 480)
Contact map shape: (273, 273)


In [25]:


# config_path = os.path.join(os.path.dirname(__file__), '..', 'config.yaml')

with open(config_path,'r') as yaml_file:
    config = yaml.safe_load(yaml_file)

merged_df_train_save_path = os.path.join(config['top_level_folder'],'data/ready_for_training/train_gt_embed_preds.pkl')
merged_df = pd.read_pickle(merged_df_train_save_path)
merged_df['seq_length'] = merged_df['residue_sequence'].apply(len)

print(merged_df.columns)

merged_df.head()

train_knn_df = merged_df.reset_index(drop=True)

Index(['filename', 'chain_number', 'residue_sequence', 'contact_map',
       'uniq_id', 'esm2_embeddings', 'esm2_contact_map_preds',
       'gt_contact_map_shapes', 'esm2_cont_preds_shapes', 'seq_length'],
      dtype='object')


In [4]:
len(merged_df[merged_df.duplicated(subset=['uniq_id'],keep=False)])

0

In [26]:
from sklearn.neighbors import NearestNeighbors
def build_knn_index(embeddings, metric="cosine"):
    """
    embeddings: np.ndarray of shape (N, d)
    metric: "cosine" or "euclidean"
    Returns: fitted NearestNeighbors model
    """
    knn = NearestNeighbors(metric=metric, algorithm="brute")  # brute-force is fine for up to ~20k vectors
    knn.fit(embeddings)
    return knn

# merged_clean_train_df = pd.read_pickle(merged_clean_train_df_path)
# merged_clean_train_df = merged_df.copy()
# uniq_ids = merged_clean_train_df['uniq_id'].values
emb_list = train_knn_df['esm2_embeddings'].values
emb_matrix = np.stack([np.asarray(e, dtype="float32").reshape(-1) for e in emb_list], axis=0)  # (N, d)

# print(uniq_ids.shape)
print(emb_list.shape)
print(emb_matrix.shape)

(12574,)
(12574, 480)


In [ ]:


def search_knn(knn, query_emb, k=5,train_embed = True):
    """
    knn: fitted NearestNeighbors model
    query_emb: np.ndarray of shape (d,)
    k: number of neighbours
    Returns: (distances, indices)
    """
    query_emb = np.asarray(query_emb, dtype="float32").reshape(1, -1)
    if train_embed == True: #if querying train embedding, embedding already present in index so we skip first entry with cosine distance 0
        distances, indices = knn.kneighbors(query_emb, n_neighbors=k+1)
        return distances[0][1:], indices[0][1:]
    else:
        distances, indices = knn.kneighbors(query_emb, n_neighbors=k)
        return distances[0], indices[0]


knn_fitted = build_knn_index(emb_matrix, metric="cosine")
all_lengths = merged_clean_train_df['seq_length'].values

def find_similar_with_length_filter(knn, cls_matrix, all_lengths, query_cls, Lq,
                                    k=5, k_candidates=20, len_filter_percent=0.2,train_query = True):
    """
    knn: fitted NearestNeighbors on CLS vectors
    cls_matrix: (N, d) matrix of CLS embeddings (only needed if knn doesn't store indices)
    all_lengths: np.ndarray of shape (N,) with training sequence lengths
    query_cls: np.ndarray of shape (d,)
    Lq: length of query sequence
    k: number of neighbours to return
    k_candidates: how many raw neighbours to ask KNN for
    min_ratio, max_ratio: allowed length range relative to query
    """
    query_cls = np.asarray(query_cls, dtype="float32").reshape(1, -1)

    # 1) get KNN candidates by cosine distance
    distances, idxs = knn.kneighbors(query_cls, n_neighbors=k_candidates)
    # idxs = idxs[0]         # shape (k_candidates,)
    # distances = distances[0]

    # 2) simple length filter
    Ls = all_lengths[idxs]
    lower = Lq * (1-len_filter_percent)
    upper = Lq * (1+len_filter_percent)
    keep_mask = (Ls >= lower) & (Ls <= upper)

    filtered_idxs = idxs[keep_mask]
    filtered_dists = distances[keep_mask]

    print(f'len filtered candidates:  {len(filtered_idxs)}')

    # 3) take first k that pass the filter
    if len(filtered_idxs) >= k:
        if train_query == True: #if comparing sequence that exists in training data, first idx   will be exact match
            return filtered_idxs[1:k+1], filtered_dists[1:k+1]
        else:

            return filtered_idxs[:k], filtered_dists[:k]
    else:
        # not enough within range → just return what we have
        return filtered_idxs, filtered_dists[:len(filtered_idxs)]


test_distances, test_indices = find_similar_with_length_filter(
    knn=knn_fitted,
    all_lengths=all_lengths,
    query_cls=test_embed,
    Lq=test_length,
    k=5, k_candidates=100,
    min_ratio=0.7, max_ratio=1.3,
)

print("indices:", test_indices)
subset_df = train_knn_df.iloc[test_indices]
print(subset_df[["uniq_id", "seq_length"]])

print(test_distances)
print(test_indices)

subset_df = merged_clean_train_df.iloc[test_indices]
subset_df.head()

len filtered candidates:  10
[0.007195   0.01661682 0.01762772 0.0181793  0.01901853]
[10960  8870  2751    41  2661]


,filename,chain_number,residue_sequence,contact_map,uniq_id,esm2_embeddings,esm2_contact_map_preds,gt_contact_map_shapes,esm2_cont_preds_shapes,seq_length
11094,6WBO.npy,1,HMRYXELAELYRRLEKTTLKTLKTKFVADFLKKTPDDLLEIVPYLI...,"[[0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",6WBO_ch_1,"[[-0.19075671, -0.15838552, 0.1684436, -0.1069...","[[0.0025901794, 3.5762787e-07, 0.0013246536, 0...","(555, 555)","(555, 555)",555
8941,4BX8.npy,1,AAHLXYGRVNLNVLREAVRRELREFLDKCAGXKAIVWDEYLTGPFG...,"[[0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,...",4BX8_ch_1,"[[-0.1472792, -0.15964614, -0.031922814, -0.00...","[[0.00057029724, 3.5762787e-07, 0.0017957687, ...","(568, 568)","(568, 568)",568
2763,1KHF.npy,1,NLXAKVVQGXLDXLPQAVREFLENNAELCQPDHIHICDGXEEENGR...,"[[0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1KHF_ch_1,"[[-0.12273673, -0.14325075, 0.008653787, 0.023...","[[0.0003554821, 4.7683716e-07, 0.0010528564, 0...","(603, 603)","(603, 603)",603
41,4CCA.npy,1,GLKAVVGEKILXGVIRXVKKDGEWKVLIMDHPXMRILXXCCKMXDI...,"[[0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",4CCA_ch_1,"[[-0.15365553, -0.2616097, -0.05210473, -0.000...","[[0.007095337, 5.9604645e-08, 0.002286911, 0.0...","(544, 544)","(544, 544)",544
2673,1FVH.npy,1,ALKTAVHEKIMNDVVLAVKKNAEWKVLIVDQLXMRMVXACCKMHEI...,"[[0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1FVH_ch_1,"[[-0.14901248, -0.21940497, -0.012243734, 0.03...","[[0.000682354, 2.7418137e-06, 0.010528564, 0.0...","(546, 546)","(546, 546)",546


In [21]:
print("test_indices:", test_indices)

subset_df = merged_clean_train_df.iloc[test_indices]

print("\nsubset_df.index (labels):", subset_df.index.to_list())
print("\nsubset_df uniq_ids + seq_length:")
print(subset_df[["uniq_id", "seq_length"]])

i = test_indices[0]  # e.g. 10960

print("\nFrom original DF with iloc:")
print(merged_clean_train_df.iloc[i][["uniq_id", "seq_length"]])

print("\nFrom subset_df (first row):")
print(subset_df.iloc[0][["uniq_id", "seq_length"]])


test_indices: [10960  8870  2751    41  2661]

subset_df.index (labels): [11094, 8941, 2763, 41, 2673]

subset_df uniq_ids + seq_length:
         uniq_id  seq_length
11094  6WBO_ch_1         555
8941   4BX8_ch_1         568
2763   1KHF_ch_1         603
41     4CCA_ch_1         544
2673   1FVH_ch_1         546

From original DF with iloc:
uniq_id       6WBO_ch_1
seq_length          555
Name: 11094, dtype: object

From subset_df (first row):
uniq_id       6WBO_ch_1
seq_length          555
Name: 11094, dtype: object


In [ ]:
print(test_distances)
print(test_indices)

#subset df


[    0 10960  8870  3251  2751]
[0.         0.007195   0.01661682 0.01662016 0.01762772]


,filename,chain_number,residue_sequence,contact_map,uniq_id,esm2_embeddings,esm2_contact_map_preds,gt_contact_map_shapes,esm2_cont_preds_shapes,seq_length
0,3GDE.npy,1,GXHMLFAEFAEFCERLEKIXXTLELTARIAAFLQKIEDERDLYDVV...,"[[0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",3GDE_ch_1,"[[-0.1946185, -0.15192227, 0.12797828, -0.1309...","[[0.0020503998, 2.3841858e-07, 0.0012741089, 0...","(551, 551)","(551, 551)",551
0,3GDE.npy,1,GXHMLFAEFAEFCERLEKIXXTLELTARIAAFLQKIEDERDLYDVV...,"[[0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",3GDE_ch_1,"[[-0.1946185, -0.15192227, 0.12797828, -0.1309...","[[0.0020503998, 2.3841858e-07, 0.0012741089, 0...","(551, 551)","(551, 551)",551
0,3GDE.npy,1,GXHMLFAEFAEFCERLEKIXXTLELTARIAAFLQKIEDERDLYDVV...,"[[0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",3GDE_ch_1,"[[-0.1946185, -0.15192227, 0.12797828, -0.1309...","[[0.0020503998, 2.3841858e-07, 0.0012741089, 0...","(551, 551)","(551, 551)",551
0,3GDE.npy,1,GXHMLFAEFAEFCERLEKIXXTLELTARIAAFLQKIEDERDLYDVV...,"[[0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",3GDE_ch_1,"[[-0.1946185, -0.15192227, 0.12797828, -0.1309...","[[0.0020503998, 2.3841858e-07, 0.0012741089, 0...","(551, 551)","(551, 551)",551
0,3GDE.npy,1,GXHMLFAEFAEFCERLEKIXXTLELTARIAAFLQKIEDERDLYDVV...,"[[0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",3GDE_ch_1,"[[-0.1946185, -0.15192227, 0.12797828, -0.1309...","[[0.0020503998, 2.3841858e-07, 0.0012741089, 0...","(551, 551)","(551, 551)",551


In [19]:

merged_df['gt_contact_map_shapes'] = merged_df['contact_map'].apply(lambda x: x.shape)
merged_df['esm2_cont_preds_shapes'] = merged_df['esm2_contact_map_preds'].apply(lambda x: x.shape)

#count rows with shape mismatch
mismatched_shape_df = merged_df[merged_df['gt_contact_map_shapes'] != merged_df['esm2_cont_preds_shapes']]

print(mismatched_shape_df.shape)

mismatched_shape_df['seq_length']  = mismatched_shape_df['residue_sequence'].apply(len)
mismatched_shape_df.head(10)

(778, 9)


/tmp/ipykernel_3133/1099402040.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mismatched_shape_df['seq_length']  = mismatched_shape_df['residue_sequence'].apply(len)


,filename,chain_number,residue_sequence,contact_map,uniq_id,esm2_embeddings,esm2_contact_map_preds,gt_contact_map_shapes,esm2_cont_preds_shapes,seq_length
310,1T4O.npy,1,TDKLDMNAKRQLYXLIGYAXLRLHYVTVKKPTAVDPNXIVECRVGD...,"[[0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1T4O_ch_1,"[[-0.53847516, 0.05677248, -0.20462482, 0.4144...","[[0.054992676, 3.0696392e-05, 0.022979736, 0.0...","(81, 81)","(82, 82)",81
313,3EVY.npy,1,MKPYEKLVERFNEMAAEFLXYFPTVKXVGNLEXELDKRRFVILFRA...,"[[0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",3EVY_ch_1,"[[-0.65856045, 0.16827904, -0.12515634, 0.5541...","[[3.170967e-05, 1.2814999e-05, 0.024230957, 0....","(83, 83)","(86, 86)",83
642,2FQ3.npy,1,XKWFNLEKIHXIEVQXLPEFFTNRIPXKTPEVYMRYRNFMVNXYRL...,"[[0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",2FQ3_ch_1,"[[-0.788006, 0.2804453, -0.3991435, 0.30526486...","[[7.969141e-05, 3.349781e-05, 0.008712769, 0.0...","(85, 85)","(86, 86)",85
974,1LPG.npy,1,RKLCXLDNGDCDQFCHEEQNXVVCXCARGYTLADNGKACIPTGPYP...,"[[0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,...",1LPG_ch_1,"[[-0.5054704, 0.30497852, -0.060636327, 0.5812...","[[0.020645142, 2.3841858e-07, 0.0013151169, 1....","(53, 53)","(82, 82)",53
1020,5MJ1.npy,1,PGTPEVKVAXXEDVDLLPCCTAPWDPQVPYTVXXWVKLLENERPYX...,"[[0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",5MJ1_ch_1,"[[-0.36805832, -0.06817822, -0.06413516, 0.254...","[[0.010169983, 0.0, 0.0071487427, 0.0015363693...","(88, 88)","(89, 89)",88
1512,5ELJ.npy,1,ENXLEIEELARFAVDEHNKKENALLEFVRVVKAKEQIDLTQEWVFT...,"[[0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",5ELJ_ch_1,"[[-0.61728746, 0.14264557, -0.06061415, 0.2615...","[[0.0018386841, 1.9073486e-06, 0.0041160583, 0...","(90, 90)","(91, 91)",90
1520,6XAT.npy,1,DVRPPFTYAXLIRQAILETPDRQLTLNEIYNWFTRMFAYFRRNTAT...,"[[0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",6XAT_ch_1,"[[-0.54650646, -0.21380146, -0.004906859, 0.39...","[[0.011329651, 2.6226044e-06, 0.0013990402, 0....","(77, 77)","(82, 82)",77
1753,5YM8.npy,1,NVFTAQNTAQDFNGNEXTVKXFYVTRTGKKILVAITXTKDNLKTVT...,"[[0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",5YM8_ch_1,"[[-0.41584915, 0.15606037, -0.45447797, 0.4690...","[[0.00020992756, 2.9802322e-07, 0.00021314621,...","(93, 93)","(94, 94)",93
2286,2C3V.npy,1,DATDITIYYKTGWTHPHIHYXLNQGAWTTLPGVPLTKXEEGVKVTI...,"[[0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",2C3V_ch_1,"[[-0.4008884, -0.105265304, -0.29222938, 0.352...","[[0.06707764, 5.9604645e-08, 0.021209717, 0.00...","(93, 93)","(94, 94)",93
2292,4ZKA.npy,1,ENKXQPKRLHVXNIPFRFRDPDLRQMFGQFGKILDVEIIFNERGXK...,"[[0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",4ZKA_ch_1,"[[-0.5336371, 0.20990777, -0.17248486, 0.28957...","[[0.20947266, 2.9802322e-05, 0.119628906, 0.02...","(84, 84)","(86, 86)",84


In [23]:
from sklearn.metrics import matthews_corrcoef

import numpy as np
import pandas as pd
from sklearn.metrics import matthews_corrcoef

def generate_baseline_metric_per_row(df, y_true_col, y_pred_col, bin_threshold=0.5):
    """
    Compute Matthews Correlation Coefficient (MCC) for each row of a DataFrame,
    where each row contains a contact map (2D array) in both true and predicted columns.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing columns with ground truth and predicted contact maps.
    y_true_col : str
        Column name containing the ground-truth binary contact maps.
    y_pred_col : str
        Column name containing the predicted probability contact maps.
    bin_threshold : float, optional
        Threshold to binarise predicted probabilities.

    Returns
    -------
    pd.Series
        MCC per row.
    float
        Average MCC across all rows.
    """
    mcc_scores = []

    for _, row in df.iterrows():
        y_true = np.array(row[y_true_col]).ravel()
        y_pred = (np.array(row[y_pred_col]) >= bin_threshold).astype(int).ravel()

        # Handle potential all-zero cases safely
        if len(np.unique(y_true)) == 1:
            mcc = np.nan  # undefined when all true values are the same
        else:
            mcc = matthews_corrcoef(y_true, y_pred)

        mcc_scores.append(mcc)

    df["mcc"] = mcc_scores
    avg_mcc = np.nanmean(mcc_scores)

    return df["mcc"], avg_mcc

mismatched_shape_df = merged_df[merged_df['gt_contact_map_shapes'] != merged_df['esm2_cont_preds_shapes']]
merged_df_bad_rows_dropped = merged_df.drop(mismatched_shape_df.index)

merged_df_bad_rows_dropped["mcc"] , avg_mcc = generate_baseline_metric_per_row(
    merged_df_bad_rows_dropped,
    y_true_col="contact_map",
    y_pred_col="esm2_contact_map_preds",
    bin_threshold=0.5
)


# 



In [25]:
merged_df_bad_rows_dropped["mcc"].describe()

count    12574.000000
mean         0.120442
std          0.066715
min         -0.016983
25%          0.070910
50%          0.116998
75%          0.165749
max          0.411680
Name: mcc, dtype: float64